# Lesson 03: Fairness & Responsible Machine Learning with Fairlearn

In this Colab notebook, we replace proprietary dashboards with open-source **`fairlearn`** and `scikit-learn` to measure selection-rate and error disparities across demographic subgroups.

In [ ]:
!pip install -q fairlearn scikit-learn pandas

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_negative_rate

rng = np.random.default_rng(7)
n = 400
group = rng.choice(["Group_A", "Group_B"], size=n, p=[0.6, 0.4])
income = rng.normal(60, 15, size=n) - np.where(group == "Group_B", 8, 0)
credit_score = rng.normal(680, 50, size=n)
y_true = ((income * 0.6 + credit_score * 0.08) > 88).astype(int)

X = pd.DataFrame({"income": income, "credit_score": credit_score, "is_group_b": (group == "Group_B").astype(int)})
clf = LogisticRegression().fit(X, y_true)
y_pred = clf.predict(X)

mf = MetricFrame(
    metrics={
        "accuracy": accuracy_score,
        "selection_rate": selection_rate,
        "tpr": true_positive_rate,
        "fnr": false_negative_rate,
    },
    y_true=y_true,
    y_pred=y_pred,
    sensitive_features=group,
)
print(mf.by_group)
print("\nDisparity differences:\n", mf.difference())

## Assignment: Responsible ML Impact Assessment

Remove `is_group_b` from `X` above, retrain `LogisticRegression`, and re-run `MetricFrame`. Does removing the sensitive column eliminate disparity, or do correlated proxy features (`income`) still create differences?